> title : 건설공사 사고 예방 및 대응책 생성 : 한솔데코 시즌3 AI 경진대회 <br>
> author : hjy <br>

- https://dacon.io/competitions/official/236455/overview/description
- https://github.com/dmskorea/project7-Hansol-Deco-Season-3-AI-Competition

<img src="https://dacon.s3.ap-northeast-2.amazonaws.com/competition/236455/header_background.jpeg" style="width:100%; height:auto;">


In [1]:
!pip uninstall -y scipy
!pip install -q scipy==1.13.0
!pip install -q -U gensim --no-deps
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U datasets ipywidgets

Found existing installation: scipy 1.13.0
Uninstalling scipy-1.13.0:
  Successfully uninstalled scipy-1.13.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
import transformers
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
from datasets import load_dataset
from peft import prepare_model_for_kbit_training,LoraConfig,PeftModel,get_peft_model

from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig
fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
    optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
)
accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

In [4]:
import pandas as pd
import numpy as np
import os
import sys
import re

In [5]:
pd.set_option("display.max_columns", None)  # 모든 컬럼 출력
pd.set_option("display.max_colwidth", 20)  # 컬럼 내용 생략 없음
pd.set_option('display.max_rows', 100)

#### read data

In [6]:
# path = '/kaggle/input/project7'
path = '/content/drive/MyDrive'

# [1]
train = pd.read_csv(f'{path}/train_new.csv', encoding = 'utf-8-sig')
valid = pd.read_csv(f'{path}/valid.csv', encoding = 'utf-8-sig')
test = pd.read_csv(f'{path}/test_new.csv', encoding = 'utf-8-sig')
submission = pd.read_csv(f'{path}/sample_submission.csv', encoding = 'utf-8-sig')

#### 데이터 전처리

In [7]:
# 장소
train['장소(대분류)'] = train['장소'].str.split('/').str[0].str.strip()
train['장소(중분류)'] = train['장소'].str.split('/').str[1].str.strip()
valid['장소(대분류)'] = valid['장소'].str.split('/').str[0].str.strip()
valid['장소(중분류)'] = valid['장소'].str.split('/').str[1].str.strip()
test['장소(대분류)'] = test['장소'].str.split('/').str[0].str.strip()
test['장소(중분류)'] = test['장소'].str.split('/').str[1].str.strip()

# 부위
train['부위(대분류)'] = train['부위'].str.split('/').str[0].str.strip()
train['부위(중분류)'] = train['부위'].str.split('/').str[1].str.strip()
valid['부위(대분류)'] = valid['부위'].str.split('/').str[0].str.strip()
valid['부위(중분류)'] = valid['부위'].str.split('/').str[1].str.strip()
test['부위(대분류)'] = test['부위'].str.split('/').str[0].str.strip()
test['부위(중분류)'] = test['부위'].str.split('/').str[1].str.strip()

# 발생일시
def preprocess_datetime(s):
    import datetime
    d, m, t = s.split()
    dt = d+" "+t
    dt = datetime.datetime.strptime(dt, "%Y-%m-%d %H:%M")
    if m == "오후" and dt.hour != 12:
        dt = dt + datetime.timedelta(hours = 12)
    return dt

train['발생일시'] = train['발생일시'].map(preprocess_datetime)
valid['발생일시'] = valid['발생일시'].map(preprocess_datetime)
test['발생일시'] = test['발생일시'].map(preprocess_datetime)

train['사고발생시간'] = train['발생일시'].dt.hour
valid['사고발생시간'] = valid['발생일시'].dt.hour
test['사고발생시간'] = test['발생일시'].dt.hour

weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
train['사고발생요일'] = train['발생일시'].dt.day_of_week.map(weekday_map)
valid['사고발생요일'] = valid['발생일시'].dt.day_of_week.map(weekday_map)
test['사고발생요일'] = test['발생일시'].dt.day_of_week.map(weekday_map)

train['사고발생월'] = train['발생일시'].dt.month
valid['사고발생월'] = valid['발생일시'].dt.month
test['사고발생월'] = test['발생일시'].dt.month

In [8]:
combined_train_data = train.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "context": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
        ),
        "answer": row["재발방지대책"] # 재발방지대책 및 향후조치계획
    },
    axis=1
)

combined_valid_data = valid.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "context": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
        ),
        "answer": row["재발방지대책"] # 재발방지대책 및 향후조치계획
    },
    axis=1
)

combined_test_data = test.apply(
    lambda row: {
        "question": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
            f"재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
        ),
        "context": (
            f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중 "
            f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서 "
            f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다. "
            f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형 입니다. "
            f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다. "
            f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서 "
            f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다. "
            f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다. "
            f"사고발생시간은 {row['사고발생시간']}시 이며, 사고요일은 {row['사고발생요일']}요일 이고, 사고발생월은 {row['사고발생월']}월 입니다. "
        )
    },
    axis=1
)

# DataFrame으로 변환
combined_train_data = pd.DataFrame(list(combined_train_data))
combined_valid_data = pd.DataFrame(list(combined_valid_data))
combined_test_data = pd.DataFrame(list(combined_test_data))

In [9]:
def create_context(row, target_words):
    """
    사고 데이터에서 주요 정보를 포함하는 문장을 생성하는 함수.
    - 사전 전처리를 적용하여, 타겟 데이터(test)에서 존재하지 않는 값은 None으로 변경.
    - None, 'nan', '' 값이 포함된 문장은 제거하여 문장을 깔끔하게 유지.

    Parameters:
        row (pd.Series): 데이터셋의 한 행
        target_words (dict): 각 열별 허용된 단어 목록 (test 데이터 기반)

    Returns:
        str: 필터링된 문장
    """
    # 사전 전처리: `test` 기준으로 존재하는 값만 유지
    전처리대상후보변수 = [
        '공사종류', '인적사고', '물적사고', '공종', '사고객체', '작업프로세스', '장소', '부위', '사고원인',
        '공사종류(대분류)', '공사종류(중분류)', '공종(대분류)', '공종(중분류)', '사고객체(대분류)', '사고객체(중분류)',
        '장소(대분류)', '장소(중분류)', '부위(대분류)', '부위(중분류)'
    ]

    for col in 전처리대상후보변수:
        if row[col] not in target_words[col]:  # `test` 기준 값이 아니면 None 처리
            row[col] = None

    # 문장 생성
    sentences = [
        f"공사종류 대분류 '{row['공사종류(대분류)']}', 중분류 '{row['공사종류(중분류)']}' 공사 중",
        f"공종 대분류 '{row['공종(대분류)']}', 중분류 '{row['공종(중분류)']}' 작업에서",
        f"사고객체 '{row['사고객체(대분류)']}'(중분류: '{row['사고객체(중분류)']}')와 관련된 사고가 발생했습니다.",
        f"작업 프로세스는 '{row['작업프로세스']}'이며, 사고 원인은 '{row['사고원인']}'이고 사고 원인 유형은 '{row['사고원인유형']}' 유형입니다.",
        f"조사결과 본 사고의 인적사고유형은 '{row['인적사고']}'이며, 물적사고유형은 '{row['물적사고']}'입니다.",
        f"장소 대분류 '{row['장소(대분류)']}', 장소 중분류 '{row['장소(중분류)']}'에서",
        f"부위 대분류 '{row['부위(대분류)']}', 부위 중분류 '{row['부위(중분류)']}' 사고가 발생하였습니다.",
        f"해당 사고 발생 당시 기온은 '{row['기온']}'이며, 발생일시는 '{row['발생일시']}'입니다.",
        f"사고발생시간은 {row['사고발생시간']}시이며, 사고요일은 {row['사고발생요일']}요일이고, 사고발생월은 {row['사고발생월']}월입니다.",
        "재발 방지 대책 및 향후 조치 계획은 무엇인가요?"
    ]

    # None, 'nan', '' 값이 포함된 문장 필터링
    filtered_sentences = [s for s in sentences if not any(v in s for v in ["None", "nan", "''"])]

    # 문장 합치기
    return " ".join(filtered_sentences)


### `test` 기준 사전 전처리 목록 생성
target_words = {col: set(test[col].dropna().unique()) for col in [
    '공사종류', '인적사고', '물적사고', '공종', '사고객체', '작업프로세스', '장소', '부위', '사고원인',
    '공사종류(대분류)', '공사종류(중분류)', '공종(대분류)', '공종(중분류)', '사고객체(대분류)', '사고객체(중분류)',
    '장소(대분류)', '장소(중분류)', '부위(대분류)', '부위(중분류)'
]}

### 전처리 포함된 `create_context` 함수 적용
combined_train_data['context'] = train.apply(lambda row: create_context(row, target_words), axis=1)
combined_valid_data['context'] = valid.apply(lambda row: create_context(row, target_words), axis=1)
combined_test_data['context'] = test.apply(lambda row: create_context(row, target_words), axis=1)

In [10]:
from datasets import Dataset

"""
Dataset({
    features: ['gem_id', 'meaning_representation', 'target', 'references'],
    num_rows: 5103
})

- gem_id : index
- meaning_representation : answer
- target : question
- references : context

"""

train_dataset = Dataset.from_pandas(combined_train_data.reset_index())
eval_dataset = Dataset.from_pandas(combined_valid_data.reset_index())
test_dataset = Dataset.from_pandas(combined_test_data.reset_index())

#### load model

In [11]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from transformers import BitsAndBytesConfig

model_id = "MLP-KTLim/llama-3-Korean-Bllossom-8B"
model_max_length = 512

# 1. 8-bit 양자화 설정 (bnb_config)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 8-bit 양자화 적용
    bnb_8bit_compute_dtype=torch.float16,  # 연산은 FP16으로 진행
    bnb_8bit_use_double_quant=True,  # 더블 양자화 적용 (더 안정적)
)

# 2. 모델 로드 (8-bit 양자화 적용)
drive_path = "/content/drive/MyDrive/models/"

if not os.path.exists(f"{drive_path}{model_id}"):
    print("모델 다운로드 중...")
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
    tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=model_max_length, padding_side="left", add_eos_token=True)

    # 모델 저장
    model.save_pretrained(f"{drive_path}{model_id}")
    tokenizer.save_pretrained(f"{drive_path}{model_id}")
    print("모델 저장 완료!")
else:
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
    tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=model_max_length, padding_side="left", add_eos_token=True)
    print("모델이 이미 저장되어 있습니다.")

# 3. Gradient Checkpointing 적용 (VRAM 절약)
model.gradient_checkpointing_enable()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

모델이 이미 저장되어 있습니다.


#### tokenizer
- gem_id : index
- meaning_representation : answer
- target : question
- references : context

In [12]:
%%time

# CPU times: user 1min 21s, sys: 14.6 s, total: 1min 36s
# Wall time: 1min 3s

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    model_max_length=model_max_length,
    padding_side="left",
    add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=model_max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

def generate_and_tokenize_prompt(data_point):
    full_prompt =f"""
    ### 지침: 당신은 건설 안전 전문가입니다.
    - 건설현장에서 사고가 발생했습니다. 주어진 [사고정보]을 기반으로, 답변(재발 방지 대책 및 향후 조치 계획)을 작성하세요.

    ### 답변(재발 방지 대책 및 향후 조치 계획) 작성 양식
    - 베스트 [재발 방지 대책 및 향후 조치 계획] 예제를 참고하여 일관된 형식으로 답변하세요.
    - 존댓말을 절대 사용하지 마세요.
    - 질문에 대한 핵심 내용만 요약하여 간략하게 핵심만 작성하세요.
    - 프롬프트에 제공된 내용을 절대로 복사하지 마세요.
    - 답변은 최대 70자 내외로 작성하세요.

    ### 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
    1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
    2. 이동통로 확보 관리와 작업 전 안전교육 철저 및 정기적 근로자 안전교육 시행을 통한 재발 방지 대책.
    3. 작업 시 안전교육 및 보호구 착용과 안전매트 설치를 통한 재발 방지 대책 및 향후 조치 계획.

    ### 사고정보:
    {data_point['context']}

    ### 질문:
    {data_point['question']}

    ### 답변(재발 방지 대책 및 향후 조치 계획):
    {data_point['answer']}
    """
    return tokenize(full_prompt)


tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = eval_dataset.map(generate_and_tokenize_prompt)

# check
print("# question: " + train_dataset[1]['question'])
print("# answer: " + train_dataset[1]['answer'] + "\n")

Map:   0%|          | 0/23198 [00:00<?, ? examples/s]

Map:   0%|          | 0/224 [00:00<?, ? examples/s]

# question: 공사종류 대분류 '건축', 중분류 '건축물' 공사 중 공종 대분류 '기계설비', 중분류 '기계설비공사' 작업에서 사고객체 '가시설'(중분류: '거푸집')와 관련된 사고가 발생했습니다. 작업 프로세스는 '설비작업'이며, 사고 원인은 '1. 작업 전 주변 위험요인 파악 후 작업하였으나, 타공정 작업자와의 소통 부재2. 알폼 작업자와의 작업 위치 미협의로 인한 혼재된 공간에서의 작업 진행 중 사고'이고 사고 원인 유형은 '미분류' 유형 입니다. 조사결과 본 사고의 인적사고유형은 '물체에 맞음'이며, 물적사고유형은 '없음'입니다. 장소 대분류 '공동주택', 장소 중분류 '내부'에서 부위 대분류 '거푸집', 부위 중분류 '옆' 사고가 발생하였습니다. 해당 사고 발생 당시 기온은 '1℃'이며, 발생일시는 '2023-12-18 14:00:00'입니다. 사고발생시간은 14시 이며, 사고요일은 월요일 이고, 사고발생월은 12월 입니다. 재발 방지 대책 및 향후 조치 계획은 무엇인가요?
# answer: 작업 전 주변 위험 요인 파악 및 정리, 주변 타공정 작업 시 위험 상황 공유, 안전 교육 철저 및 현장 수시 확인을 통한 안전 관리 강화.

CPU times: user 58.5 s, sys: 495 ms, total: 59 s
Wall time: 58.7 s


#### eval_prompt

In [13]:
eval_prompt = """
    ### 지침: 당신은 건설 안전 전문가입니다.
    - 건설현장에서 사고가 발생했습니다. 주어진 [사고정보]을 기반으로, 답변(재발 방지 대책 및 향후 조치 계획)을 작성하세요.

    ### 답변(재발 방지 대책 및 향후 조치 계획) 작성 양식
    - 베스트 [재발 방지 대책 및 향후 조치 계획] 예제를 참고하여 일관된 형식으로 답변하세요.
    - 존댓말을 절대 사용하지 마세요.
    - 질문에 대한 핵심 내용만 요약하여 간략하게 핵심만 작성하세요.
    - 프롬프트에 제공된 내용을 절대로 복사하지 마세요.
    - 답변은 최대 70자 내외로 작성하세요.

    ### 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
    1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
    2. 이동통로 확보 관리와 작업 전 안전교육 철저 및 정기적 근로자 안전교육 시행을 통한 재발 방지 대책.
    3. 작업 시 안전교육 및 보호구 착용과 안전매트 설치를 통한 재발 방지 대책 및 향후 조치 계획.

    ### 사고정보:
    {data_point['context']}

    ### 질문:
    {data_point['question']}

    ### 답변(재발 방지 대책 및 향후 조치 계획):
    """

In [14]:
device = "cuda" # the device to load the model onto
model_input = tokenizer(eval_prompt, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=model_max_length, pad_token_id=2)[0], skip_special_tokens=True))


    ### 지침: 당신은 건설 안전 전문가입니다.
    - 건설현장에서 사고가 발생했습니다. 주어진 [사고정보]을 기반으로, 답변(재발 방지 대책 및 향후 조치 계획)을 작성하세요.

    ### 답변(재발 방지 대책 및 향후 조치 계획) 작성 양식 
    - 베스트 [재발 방지 대책 및 향후 조치 계획] 예제를 참고하여 일관된 형식으로 답변하세요.  
    - 존댓말을 절대 사용하지 마세요.
    - 질문에 대한 핵심 내용만 요약하여 간략하게 핵심만 작성하세요.
    - 프롬프트에 제공된 내용을 절대로 복사하지 마세요.   
    - 답변은 최대 70자 내외로 작성하세요. 

    ### 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
    1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
    2. 이동통로 확보 관리와 작업 전 안전교육 철저 및 정기적 근로자 안전교육 시행을 통한 재발 방지 대책.
    3. 작업 시 안전교육 및 보호구 착용과 안전매트 설치를 통한 재발 방지 대책 및 향후 조치 계획.

    ### 사고정보:
    {data_point['context']}

    ### 질문:
    {data_point['question']}

    ### 답변(재발 방지 대책 및 향후 조치 계획):
     {your_answer}  # 이 곳에 답변을 작성하세요. 





```python
import pandas as pd

# 예시 데이터셋
data_point = {
    'context': '건설현장에서 작업 중인 근로자가 작업 중에 추락사고를 당했습니다.',
    'question': '이 사고를 예방하기 위한 재발 방지 대책과 향후 조치 계획을 제시해주세요.'
}

# 답변 예시
# 베스트 답변(재발 방지 대책 및 향후 조치 계획) 예시 3개:
# 1. 작업 전 안전교육 실시와 안전점검 철저 지시를 통한 재발 방지 대책 마련.
# 2. 이동통로 확보 관리와 작

In [15]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [16]:
# target_modules는 LoRA가 적용될 특정 모델의 레이어를 지정하는 옵션
# 모든 파라미터를 학습하는 것이 아니라, 특정 **레이어(모듈)**만 학습하도록 설정하여 VRAM을 절약하고 학습 속도를 높임

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",     # Query Projection (Self-Attention)
        "k_proj",     # Key Projection (Self-Attention)
        "v_proj",     # Value Projection (Self-Attention)
        "o_proj",     # Output Projection (Self-Attention)
        "gate_proj",  # Gating Mechanism in MLP
        "up_proj",    # MLP Up Projection (확장 레이어)
        "down_proj",  # MLP Down Projection (축소 레이어)
        "lm_head",    # Language Model Head (출력층)
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
print_trainable_parameters(model)
# Apply the accelerator. You can comment this out to remove the accelerator.
# model = accelerator.prepare_model(model)

trainable params: 22030336 || all params: 8052291584 || trainable%: 0.27359088739129295


In [ ]:
%%time

from tqdm import tqdm
import transformers
from transformers import TrainerCallback

class ProgressCallback(TrainerCallback):
    """Custom callback to display training progress with tqdm progress bar."""

    def __init__(self, total_steps):
        self.progress_bar = tqdm(total=total_steps, desc="Training Progress", position=0, leave=True)

    def on_step_end(self, args, state, control, **kwargs):
        """Update progress bar at the end of each step."""
        self.progress_bar.update(1)  # 1 스텝 증가

    def on_train_end(self, args, state, control, **kwargs):
        """Close progress bar when training ends."""
        self.progress_bar.close()

# 프로젝트 설정
project = "dms-project7"
base_model_name = "llama-3-Korean-Bllossom-8B"
run_name = base_model_name + "-" + project
output_dir = drive_path + run_name
print('# drive_path:', drive_path)
print('# output_dir:', output_dir)

# 토크나이저 패딩 설정
tokenizer.pad_token = tokenizer.eos_token

# 학습 옵션
batch_size = 2
num_epochs = 1
num_train_steps = (len(tokenized_train_dataset) * 1) // 2  # 총 학습 스텝 계산
max_steps = (len(tokenized_train_dataset) * num_epochs) // batch_size # max_steps
print('# num_train_steps:',num_train_steps)
print('# max_steps:',max_steps)

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=5,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,

        # num_train_epochs=1,  # max_steps 대신 epochs 설정 (1회 반복)
        max_steps = max_steps, # 학습 시간 결정 요인

        learning_rate=2.5e-5,
        logging_steps=500,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",
        save_strategy="steps",
        save_steps=500,
        evaluation_strategy="steps",
        eval_steps=500,
        do_eval=True,
        report_to='none',
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}",
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[ProgressCallback(total_steps=num_train_steps)],  # 프로그레스바 추가
)

# 파인튜닝 실행 (진행률 표시됨)
model.config.use_cache = False  # 캐시 경고 제거
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# drive_path: /content/drive/MyDrive/models/
# output_dir: /content/drive/MyDrive/models/llama-3-Korean-Bllossom-8B-dms-project7
# num_train_steps: 11599
# max_steps: 11599


Training Progress:   0%|          | 0/11599 [00:00<?, ?it/s]No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt:

Step,Training Loss,Validation Loss
500,0.202900,0.066609
1000,0.060800,0.061530
1500,0.059200,0.061131
2000,0.058700,0.059709
2500,0.058000,0.060179
3000,0.057900,0.059492
3500,0.057000,0.058858
4000,0.057000,0.058485
4500,0.056800,0.059799


Training Progress:   4%|▍         | 500/11599 [45:07<16:40:20,  5.41s/it]/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:212: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quanti

#### Load the finetuned model

In [ ]:
%%time

# CPU times: user 48.8 s, sys: 880 ms, total: 49.7 s
# Wall time: 48.7 s

# output_dir: /content/drive/MyDrive/models/llama-3-Korean-Bllossom-8B-dms-project7

ft_model = PeftModel.from_pretrained(model, f"{output_dir}/checkpoint-60")

ft_model.eval()
with torch.no_grad():
    print(tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=256, pad_token_id=2)[0], skip_special_tokens=True,repetition_penalty=1.5,temperature=0.2))